<a href="https://colab.research.google.com/github/athishsreeram/tubenotebook/blob/main/DownloadTranscripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─────────────────────────────
# Install dependencies (run once in Colab)
# ─────────────────────────────
!pip install yt-dlp youtube-transcript-api --upgrade

# ─────────────────────────────
# Imports
# ─────────────────────────────
import yt_dlp
from datetime import datetime
from youtube_transcript_api import YouTubeTranscriptApi

# ─────────────────────────────
# Configuration
# ─────────────────────────────
CHANNEL_URL = "https://www.youtube.com/@TheDiaryOfACEO/videos"  # ← Change this
MAX_VIDEOS = 80              # Total videos to fetch from channel
SKIP_FIRST_N = 38             # Skip first N videos (most recent)
DOWNLOAD_N = 38              # Number of videos to download transcripts for (after skipping)
TRANSCRIPT_LANGS = ["en"]    # Language for transcripts

# ─────────────────────────────
# Fetch channel videos
# ─────────────────────────────
def get_channel_videos(channel_url, max_videos=None):
    ydl_opts = {
        "quiet": True,
        "extract_flat": True,
        "skip_download": True,
        "ignoreerrors": True,
        "playlistend": max_videos,
    }

    videos = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        entries = info.get("entries", [])
        for entry in entries:
            if not entry:
                continue
            video_id = entry.get("id")
            videos.append({
                "title": entry.get("title"),
                "url": f"https://www.youtube.com/watch?v={video_id}",
                "video_id": video_id,
                "upload_date": entry.get("upload_date"),
            })
    return videos

# ─────────────────────────────
# Fetch transcript only
# ─────────────────────────────
def get_transcript(video_id, langs=None):
    if langs is None:
        langs = ["en"]
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=langs)
        full_text = " ".join([t.text for t in transcript])
        return full_text
    except Exception as e:
        print(f"⚠️ Transcript unavailable for {video_id}: {e}")
        return None

# ─────────────────────────────
# Save transcript as markdown
# ─────────────────────────────
def save_transcript_md(video_info, transcript_text, index):
    # Create filename
    safe_title = "".join(c for c in video_info['title'][:50] if c.isalnum() or c in (' ', '-', '_')).rstrip()
    filename = f"{index:02d}_{safe_title}.md"

    # Write markdown file
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# {video_info['title']}\n\n")
        f.write(f"**Video URL:** {video_info['url']}\n\n")
        f.write(f"**Video ID:** {video_info['video_id']}\n\n")
        f.write(f"**Upload Date:** {video_info['upload_date']}\n\n")
        f.write(f"---\n\n")
        f.write(f"## Transcript\n\n")

        if transcript_text:
            f.write(transcript_text)
        else:
            f.write("*No transcript available for this video*")

    print(f"💾 Saved: {filename}")

# ─────────────────────────────
# Run pipeline
# ─────────────────────────────
print(f"🚀 Fetching videos from {CHANNEL_URL}...")
print(f"📊 Settings: Fetch {MAX_VIDEOS} videos, Skip first {SKIP_FIRST_N}, Download next {DOWNLOAD_N}\n")

# Get all videos
all_videos = get_channel_videos(CHANNEL_URL, MAX_VIDEOS)
print(f"✅ Found {len(all_videos)} total videos\n")

# Skip first N videos (most recent)
videos_to_download = all_videos[SKIP_FIRST_N:SKIP_FIRST_N + DOWNLOAD_N]

print(f"⏭️  Skipped first {SKIP_FIRST_N} videos (most recent)")
print(f"🎯 Will download transcripts for videos {SKIP_FIRST_N + 1} to {SKIP_FIRST_N + len(videos_to_download)}\n")

if not videos_to_download:
    print("❌ No videos to download! Check your SKIP and DOWNLOAD settings.")
else:
    print(f"🎤 Getting transcripts for {len(videos_to_download)} videos...\n")

    success_count = 0
    for i, video in enumerate(videos_to_download, 1):
        actual_number = SKIP_FIRST_N + i
        print(f"[{actual_number}/{SKIP_FIRST_N + DOWNLOAD_N}] Processing: {video['title'][:60]}...")

        # Get transcript
        transcript_text = get_transcript(video["video_id"], TRANSCRIPT_LANGS)

        # Save as markdown file
        save_transcript_md(video, transcript_text, actual_number)

        if transcript_text:
            print(f"   ✅ Transcript saved ({len(transcript_text)} chars)")
            success_count += 1
        else:
            print(f"   ❌ No transcript available")
        print()

    print(f"🎯 DONE! Saved {success_count}/{len(videos_to_download)} transcript files as .md format")
    print(f"   (Skipped first {SKIP_FIRST_N} videos)")

In [ ]:
mkdir -p d
mv *.md d/

In [ ]:
import shutil
shutil.make_archive("d", "zip", "d")